In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

data=["hey what are you doing tonight",
"let me know when you are free to talk",
"i am running a bit late for the meeting",
"can you send me the file as soon as possible",
"i think we should grab some coffee later",
"happy birthday hope you have a great day",
"did you see the news this morning",
"i will be there in about ten minutes",
"let's meet at the park near the station",
"sorry i missed your call i was driving",
"do you want to watch a movie tonight",
"please remember to buy some milk on your way home",
"that sounds like a great idea to me",
"i am looking forward to seeing you again",
"call me when you get a chance",
"the weather is really nice today isn't it",
"i forgot to tell you something important",
"can we reschedule our lunch for tomorrow",
"i just finished my homework for the week",
"let me check my calendar and get back to you"]

# 1. Tokenize
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data)
total_words = len(tokenizer.word_index) + 1  # +1 for padding
print(tokenizer.word_index)


#Tokenizer() creates a tokenizer object which will convert words into integers.
#fit_on_texts(corpus) scans your list of sentences and assigns a unique integer to each unique word.
#We add +1 because word_index starts counting from 1, and we want 0 reserved for padding when we pad sequences later.

#Convert each sentence to a sequence of integers
#for every sentence, you create multiple training sequences, where the model learns


{'you': 1, 'i': 2, 'to': 3, 'the': 4, 'me': 5, 'a': 6, 'for': 7, 'are': 8, 'tonight': 9, 'let': 10, 'when': 11, 'am': 12, 'can': 13, 'as': 14, 'we': 15, 'some': 16, 'great': 17, 'your': 18, 'call': 19, 'get': 20, 'my': 21, 'hey': 22, 'what': 23, 'doing': 24, 'know': 25, 'free': 26, 'talk': 27, 'running': 28, 'bit': 29, 'late': 30, 'meeting': 31, 'send': 32, 'file': 33, 'soon': 34, 'possible': 35, 'think': 36, 'should': 37, 'grab': 38, 'coffee': 39, 'later': 40, 'happy': 41, 'birthday': 42, 'hope': 43, 'have': 44, 'day': 45, 'did': 46, 'see': 47, 'news': 48, 'this': 49, 'morning': 50, 'will': 51, 'be': 52, 'there': 53, 'in': 54, 'about': 55, 'ten': 56, 'minutes': 57, "let's": 58, 'meet': 59, 'at': 60, 'park': 61, 'near': 62, 'station': 63, 'sorry': 64, 'missed': 65, 'was': 66, 'driving': 67, 'do': 68, 'want': 69, 'watch': 70, 'movie': 71, 'please': 72, 'remember': 73, 'buy': 74, 'milk': 75, 'on': 76, 'way': 77, 'home': 78, 'that': 79, 'sounds': 80, 'like': 81, 'idea': 82, 'looking': 83,

In [4]:
input_sequences = []
for line in data:
    token_list = tokenizer.texts_to_sequences([line])[0]
    
    
    if line == data[0]:
        print(f"Original Sentence: {line}")
        print(f"Token List: {token_list}")
        print("--- Generating N-Grams ---")

    
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)
        
#Print the growth of the sentence
        if line == data[0]:
            print(n_gram_sequence)

Original Sentence: hey what are you doing tonight
Token List: [22, 23, 8, 1, 24, 9]
--- Generating N-Grams ---
[22, 23]
[22, 23, 8]
[22, 23, 8, 1]
[22, 23, 8, 1, 24]
[22, 23, 8, 1, 24, 9]


In [5]:
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

print(input_sequences.shape) 
print(input_sequences[0])


(142, 10)
[ 0  0  0  0  0  0  0  0 22 23]


In [6]:
X, y = input_sequences[:,:-1], input_sequences[:,-1]
print("Question (X):", X[0])
print("Answer (y):", y[0])


Question (X): [ 0  0  0  0  0  0  0  0 22]
Answer (y): 23


In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential()

# 1. The Embedding Layer

model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))

# 2. The LSTM Layer
# 150 units is the memory capacity of the layer
model.add(LSTM(150))

model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 9, 100)            11100     
                                                                 
 lstm (LSTM)                 (None, 150)               150600    
                                                                 
 dense (Dense)               (None, 111)               16761     
                                                                 
Total params: 178461 (697.11 KB)
Trainable params: 178461 (697.11 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [8]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=total_words)

model.fit(X, y_one_hot, epochs=100, verbose=1)

Epoch 1/100


5/5 [==============================] - 4s 11ms/step - loss: 4.7094 - accuracy: 0.0282
Epoch 2/100
5/5 [==============================] - 0s 10ms/step - loss: 4.6886 - accuracy: 0.0775
Epoch 3/100
5/5 [==============================] - 0s 10ms/step - loss: 4.6587 - accuracy: 0.0775
Epoch 4/100
5/5 [==============================] - 0s 10ms/step - loss: 4.5887 - accuracy: 0.0704
Epoch 5/100
5/5 [==============================] - 0s 10ms/step - loss: 4.4635 - accuracy: 0.0704
Epoch 6/100
5/5 [==============================] - 0s 10ms/step - loss: 4.4204 - accuracy: 0.0704
Epoch 7/100
5/5 [==============================] - 0s 8ms/step - loss: 4.3708 - accuracy: 0.0704
Epoch 8/100
5/5 [==============================] - 0s 7ms/step - loss: 4.3303 - accuracy: 0.0704
Epoch 9/100
5/5 [==============================] - 0s 11ms/step - loss: 4.3019 - accuracy: 0.0775
Epoch 10/100
5/5 [==============================] - 0s 8ms/step - loss: 4.2751 - accuracy: 0.0775
Epoch 11/100
5/5 [==

In [9]:
def predict_next_word(seed_text):
    for _ in range(1): 
        
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        
       
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        
        predicted_probs = model.predict(token_list, verbose=0)
       
        predicted_id = np.argmax(predicted_probs, axis=-1)[0]
  
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_id:
                output_word = word
                break
        return output_word

print("Suggestion for 'hey':", predict_next_word("hey"))
print("Suggestion for 'let me':", predict_next_word("let me"))

Suggestion for 'hey': what
Suggestion for 'let me': check
